# Threat Detection Demo
**Open this notebook in Google Colab and choose Runtime → Run all.** The notebook downloads the project model automatically and uses built-in samples. There is nothing to upload or edit for the default demo.

Follow the sections below from validation to image preprocessing, image detection and video playback. The last cell downloads one ZIP with your results.

In [ ]:
#@title 1. Settings · leave as is for the demo
WEIGHTS = "" #@param {type:"string"}
DATA_YAML = "" #@param {type:"string"}
SAMPLE_IMAGES = []  # Example: ["/content/photo.jpg"]
VIDEO_IN = "" #@param {type:"string"}
WEBCAM_MODE = "off" #@param ["off", "window", "inline"]
WEBCAM_SECONDS = 8 #@param {type:"integer"}
OBSERVATIONS = {}  # Example: {"Empty scene": "No detections"}

### Try your own files (optional)
In Colab, click the **Files** folder icon on the left, upload your media, then copy its path into SAMPLE_IMAGES or VIDEO_IN in Section 1. Set DATA_YAML only if you have a labeled YOLO validation dataset. Leave WEIGHTS blank to use the project model.

In [ ]:
#@title Set up the detector
import importlib.metadata
import subprocess
import sys
from pathlib import Path

required = "8.4.176"
try:
    installed = importlib.metadata.version("ultralytics")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != required:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"ultralytics=={required}"])
    if "ultralytics" in sys.modules:
        raise RuntimeError("Restart the session, then run all cells again.")

checkpoint = Path("threat_detection_baseline_m960_v2.pt")
print("Setup complete.")

In [ ]:
#@title Load the trained model
import base64
import gzip
import hashlib
import json
from io import BytesIO
from urllib.request import urlopen

try:
    from google.colab import files
except ImportError:
    files = None

checkpoint = Path(WEIGHTS).expanduser() if WEIGHTS else Path("threat_detection_baseline_m960_v2.pt")
expected_hash = "19e8e8e3bfd155da484d5a62984337392edd20d22ac4ea02e7998c504edcdac1"

if not WEIGHTS:
    if not checkpoint.is_file() or hashlib.sha256(checkpoint.read_bytes()).hexdigest() != expected_hash:
        local = Path("Threat_Detection_Simple_Demo.ipynb")
        if local.is_file():
            with local.open(encoding="utf-8") as handle:
                parts = json.load(handle)["metadata"]["embedded_checkpoint"]
        else:
            url = ("https://raw.githubusercontent.com/Doffcode/threat-detection/"
                   "8faedfb73a0e5ea43bd7f0ce2259dfcf15307388/demo/Threat_Detection_Demo.ipynb")
            print("Downloading the trained model. This can take a minute.")
            with urlopen(url, timeout=90) as handle:
                parts = json.load(handle)["metadata"]["embedded_checkpoint"]
        with BytesIO() as packed:
            for part in parts:
                packed.write(base64.b85decode(part))
            packed.seek(0)
            with gzip.GzipFile(fileobj=packed) as stream:
                weight_bytes = stream.read()
        if hashlib.sha256(weight_bytes).hexdigest() != expected_hash:
            raise ValueError("The checkpoint failed its integrity check.")
        checkpoint.write_bytes(weight_bytes)
        del parts, weight_bytes
    print("Baseline model loaded and verified.")
elif not checkpoint.is_file():
    raise FileNotFoundError(checkpoint)

In [ ]:
#@title 2. Load media and model
import csv
import math
import shutil
import tempfile
import time
from collections import Counter
from zipfile import ZipFile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Video, clear_output, display
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox


sample_dir = Path(tempfile.mkdtemp(prefix="threat_samples_"))
sample = sample_dir / "sample_handgun.jpg"
sample.write_bytes(base64.b64decode(('...')))
images = [Path(p).expanduser() for p in SAMPLE_IMAGES] if SAMPLE_IMAGES else [sample]
if VIDEO_IN:
    videos = [Path(VIDEO_IN).expanduser()]
else:
    sample_video = sample_dir / "sample_motion_demo.mp4"
    original = cv2.resize(cv2.imread(str(sample)), (640,640))
    writer = cv2.VideoWriter(str(sample_video),cv2.VideoWriter_fourcc(*"mp4v"),4,(640,640))
    if not writer.isOpened():
        raise OSError("Could not create the built-in video.")
    try:
        for step in range(12):
            transform = np.float32([[1,0,step-6],[0,1,0]])
            writer.write(cv2.warpAffine(original,transform,(640,640),borderMode=cv2.BORDER_REPLICATE))
    finally:
        writer.release()
    videos = [sample_video]

model = YOLO(str(checkpoint))
labels = {0: "handgun", 1: "rifle", 2: "knife"}
assert model.names == labels, model.names
output = Path(tempfile.mkdtemp(prefix="threat_demo_", dir=Path.cwd()))
settings = dict(imgsz=960, conf=0.30, iou=0.50, classes=list(labels), device="cpu", verbose=False)
columns = ["frame_index", "class", "confidence", "x1", "y1", "x2", "y2"]

def predict_frame(frame, index):
    start = time.perf_counter()
    result = model.predict(frame, **settings)[0]
    milliseconds = (time.perf_counter() - start) * 1000
    rows = [
        [index, result.names[int(cls)], round(conf, 6), *[round(v, 2) for v in (x1, y1, x2, y2)]]
        for x1, y1, x2, y2, conf, cls in result.boxes.data.cpu().tolist()
    ]
    return result.plot(), rows, milliseconds

def show_frame(axis, frame, title):
    axis.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axis.set_title(title)
    axis.axis("off")

def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(output / name, dpi=130)
    display(fig)
    plt.close(fig)

## 2. How well does the detector perform?
See overall and per-class validation metrics, the confusion matrix, and the PR and F1 curves. The defaults show saved project results. Set DATA_YAML for a fresh labeled evaluation.

In [ ]:
#@title Show validation evidence
metric_columns = ["class", "precision", "recall", "f1", "mAP50", "mAP50-95"]
reference = pd.DataFrame([['overall', 0.8162014976174268, 0.7877792378449409, 0.8017385489802742, 0.8238161201335835, 0.5484985953018895], ['handgun', 0.8881685575364667, 0.7884892086330936, 0.8353658536585366, 0.8568025300214289, 0.635051423161938], ['rifle', 0.7522441651705566, 0.8151750972762646, 0.7824463118580766, 0.8298595929340297, 0.4886573847779767], ['knife', 0.7864406779661017, 0.7412140575079872, 0.7631578947368421, 0.7847862374452917, 0.521786977965754]], columns=metric_columns)
print("Saved baseline validation results, 1,453 images. Precision, recall and F1 use confidence 0.30.")
display(reference.round(4))
reference.to_csv(output / "reference_validation.csv", index=False)

if DATA_YAML:
    data = Path(DATA_YAML).expanduser().resolve()
    if not data.is_file():
        raise FileNotFoundError(data)
    from ultralytics.data.utils import check_det_dataset
    dataset = check_det_dataset(str(data), autodownload=False)
    if dataset["names"] != labels:
        raise ValueError("The dataset YAML must use handgun, rifle, knife in that order.")
    metrics = model.val(data=str(data), split="val", imgsz=960, device="cpu",
                        workers=0, batch=4, plots=True, project=str(output), name="validation")
    box = metrics.box
    def f1(p, r):
        return 2*p*r/(p+r) if p+r else 0.0
    overall_f1 = float(np.mean([f1(p, r) for p, r in zip(box.p, box.r)])) if len(box.p) else 0.0
    rows = [["overall (macro)", box.mp, box.mr, overall_f1, box.map50, box.map]]
    for i, cls in enumerate(box.ap_class_index):
        p, r, ap50, ap = box.class_result(i)
        rows.append([labels[int(cls)], p, r, f1(p,r), ap50, ap])
    present = {row[0] for row in rows}
    rows.extend([name, None, None, None, None, None] for name in labels.values() if name not in present)
    table = pd.DataFrame(rows, columns=metric_columns)
    print("Fresh Ultralytics validation. Precision, recall and F1 use its selected operating point.")
    display(table.round(4))
    table.to_csv(output / "validation_metrics.csv", index=False)
    from IPython.display import Image as NotebookImage
    for filename in ("confusion_matrix.png", "BoxPR_curve.png", "BoxF1_curve.png"):
        path = Path(metrics.save_dir) / filename
        if path.exists():
            display(NotebookImage(filename=str(path)))
else:
    from IPython.display import Image as NotebookImage
    print("Saved training-run plots for the baseline validation split.")
    for filename, data in {'confusion_matrix.png':('...'),'BoxPR_curve.png':('...'),'BoxF1_curve.png':('...')}.items():
        path = output / filename
        path.write_bytes(base64.b64decode(data))
        display(NotebookImage(filename=str(path)))

## 3. What does preprocessing do?
Compare original, denoised, contrast-enhanced and letterboxed images, then see the histogram and normalized tensor stats. The displayed enhancements are illustrative. Detection uses the original frame with the model’s built-in resizing and normalization.

In [ ]:
#@title Show preprocessing stages
preprocessing_rows = []
for index, path in enumerate(images):
    frame = cv2.imread(str(path))
    if frame is None:
        print("Could not read", path)
        continue
    stem = f"image_{index+1:02d}"
    denoised = cv2.GaussianBlur(frame, (3,3), 0)
    lab = cv2.cvtColor(denoised, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(lab[:, :, 0])
    enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    letterbox = LetterBox(new_shape=(960,960), auto=True, stride=32)
    resized = letterbox(image=enhanced)
    tensor = np.ascontiguousarray(resized[..., ::-1].transpose(2,0,1))[None].astype(np.float32) / 255
    original_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    enhanced_gray = cv2.cvtColor(enhanced, cv2.COLOR_BGR2GRAY)

    print(path.name, "preprocessing demonstration")
    print(f"Grayscale standard deviation {original_gray.std():.2f} → {enhanced_gray.std():.2f}")
    print(f"Normalized tensor shape {tensor.shape}, min {tensor.min():.3f}, max {tensor.max():.3f}, mean {tensor.mean():.3f}, std {tensor.std():.3f}")
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    for ax, img, label in zip(axes, [frame,denoised,enhanced,resized], ["Original","Denoised","CLAHE","Letterboxed"]):
        show_frame(ax,img,label)
    save_figure(fig,f"{stem}_preprocessing.png")
    fig, ax = plt.subplots(figsize=(7, 3))
    for gray, label in [(original_gray,"Original"),(enhanced_gray,"CLAHE")]:
        ax.plot(np.bincount(gray.ravel(), minlength=256)/gray.size,label=label)
    ax.set(xlabel="Gray level",ylabel="Pixel fraction")
    ax.legend()
    save_figure(fig,f"{stem}_histogram.png")


    preprocessing_rows.append([path.name,original_gray.std(),enhanced_gray.std(),
                               *tensor.shape,tensor.min(),tensor.max(),tensor.mean(),tensor.std()])

pd.DataFrame(preprocessing_rows,columns=["source","original_gray_std","clahe_gray_std",
             "batch","channels","height","width","tensor_min","tensor_max","tensor_mean","tensor_std"]
             ).to_csv(output/"preprocessing_stats.csv",index=False)

## 4. What did the model detect?
Compare input and annotated output. See labels, confidences, box coordinates and the warmed latency and FPS benchmark.

In [ ]:
#@title Run image detection
image_rows = []
timing_rows = []
for index, path in enumerate(images):
    frame = cv2.imread(str(path))
    if frame is None:
        continue
    stem = f"image_{index+1:02d}"
    # Denoising and CLAHE are illustrated above. The frozen detector receives
    # the original frame and performs its own letterbox and normalization.
    model.predict(frame, **settings)
    annotated, rows, ms = predict_frame(frame,0)
    image_rows.extend([[path.name,*row] for row in rows])
    timing_rows.append([path.name,ms,1000/ms])
    cv2.imwrite(str(output / f"{stem}_annotated.png"), annotated)
    fig, axes = plt.subplots(1,2,figsize=(12,5))
    show_frame(axes[0],frame,"Input")
    show_frame(axes[1],annotated,f"Predictions · {ms:.1f} ms")
    save_figure(fig,f"{stem}_comparison.png")

if timing_rows:
    table = pd.DataFrame(image_rows,columns=["source",*columns])
    display(table)
    table.to_csv(output/"image_detections.csv",index=False)
    timing = pd.DataFrame(timing_rows,columns=["source","latency_ms","equivalent_fps"])
    display(timing.round(2))
    timing.to_csv(output/"image_timing.csv",index=False)
    fig, axes = plt.subplots(1,2,figsize=(9,3))
    indices = np.arange(len(timing))+1
    axes[0].bar(indices,timing.latency_ms)
    axes[0].set(xlabel="Sample image",ylabel="Latency (ms)")
    axes[1].bar(indices,timing.equivalent_fps)
    axes[1].set(xlabel="Sample image",ylabel="Equivalent FPS")
    save_figure(fig,"image_timing.png")
else:
    print("Provide a sample image to see preprocessing and detections.")

## 5. How does it work on video?
The built-in clip pans across the sample image so you can see frame-by-frame predictions. See the annotated video, class counts, processing FPS, sample frames and detection timeline. Set VIDEO_IN to try your own footage.

In [ ]:
#@title Run video detection
for index, path in enumerate(videos):
    stem = f"video_{index+1:02d}"
    media_path = output / f"{stem}_annotated.mp4"
    capture = cv2.VideoCapture(str(path))
    video = None
    counts = Counter()
    per_frame = []
    preview_frames = []
    start = time.perf_counter()
    try:
        readable, frame = capture.read()
        if not readable:
            print("Could not decode", path)
            continue
        fps = capture.get(cv2.CAP_PROP_FPS)
        if not math.isfinite(fps) or fps <= 0:
            fps = 25
        height, width = frame.shape[:2]
        video = cv2.VideoWriter(str(media_path), cv2.VideoWriter_fourcc(*"mp4v"), fps,
                                (width + width%2, height + height%2))
        if not video.isOpened():
            raise OSError("Could not open the MP4 writer.")
        with (output / f"{stem}_detections.csv").open("w",newline="") as handle:
            writer = csv.writer(handle)
            writer.writerow(columns)
            while readable:
                annotated, rows, ms = predict_frame(frame,len(per_frame))
                writer.writerows(rows)
                counts.update(row[1] for row in rows)
                if len(preview_frames) < 3 and (not per_frame or len(per_frame) % max(1,round(fps)) == 0):
                    preview_frames.append((len(per_frame),annotated.copy()))
                video.write(cv2.copyMakeBorder(annotated,0,height%2,0,width%2,cv2.BORDER_CONSTANT))
                per_frame.append([len(per_frame),len(rows),ms])
                readable, frame = capture.read()
    finally:
        capture.release()
        if video is not None:
            video.release()
    elapsed = time.perf_counter()-start
    summary = pd.DataFrame({"class":list(labels.values()),"detections":[counts[name] for name in labels.values()]})
    print(f"{path.name} · {len(per_frame)} frames · source {fps:.1f} FPS · processed {len(per_frame)/elapsed:.1f} FPS")
    print("Counts are predictions across frames. An object visible in consecutive frames contributes once per frame.")
    display(summary)
    summary.to_csv(output/f"{stem}_counts.csv",index=False)
    timeline = pd.DataFrame(per_frame,columns=["frame_index","detections","latency_ms"])
    timeline.to_csv(output/f"{stem}_timeline.csv",index=False)
    pd.DataFrame([[fps,len(per_frame)/elapsed,len(per_frame),elapsed]],
                 columns=["source_fps","processing_fps","frames","seconds"]).to_csv(output/f"{stem}_timing.csv",index=False)
    fig, ax = plt.subplots(figsize=(9,3))
    ax.plot(timeline.frame_index,timeline.detections,marker=".",markersize=3)
    ax.set(xlabel="Frame index",ylabel="Detections")
    save_figure(fig,f"{stem}_timeline.png")
    fig, axes = plt.subplots(1,len(preview_frames),figsize=(12,4),squeeze=False)
    for ax,(i,frame) in zip(axes[0],preview_frames):
        show_frame(ax,frame,f"Frame {i}")
    save_figure(fig,f"{stem}_preview.png")
    if shutil.which("ffmpeg"):
        converted = output / f"{stem}_h264.mp4"
        run = subprocess.run(["ffmpeg","-nostdin","-y","-loglevel","error","-i",str(media_path),
                              "-c:v","libx264","-pix_fmt","yuv420p","-movflags","+faststart",str(converted)],
                             capture_output=True)
        if run.returncode == 0:
            converted.replace(media_path)
            if media_path.stat().st_size <= 8_000_000:
                display(Video(str(media_path),embed=True))
if not videos:
    print("Provide a short video to see the annotated playback and frame timeline.")

## 6. Optional webcam test
Set WEBCAM_MODE to inline for a fixed-duration test that saves the first detection. Use window on a local desktop. Press Q to quit or S to save a snapshot.

In [ ]:
#@title Run the optional webcam test
if WEBCAM_MODE != "off":
    if files is not None and WEBCAM_MODE == "window":
        print("Window mode needs a local desktop. Use inline in Colab.")
    elif files is not None:
        from google.colab.output import eval_js
        import base64
        frames = eval_js("""(async (seconds) => {
          const stream = await navigator.mediaDevices.getUserMedia({video:{width:640,height:480}});
          const video = document.createElement('video');
          video.srcObject = stream;
          video.muted = true;
          document.body.appendChild(video);
          const canvas = document.createElement('canvas');
          const captured = [];
          try {
            await video.play();
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            const context = canvas.getContext('2d');
            for (let i=0; i<seconds; ++i) {
              context.drawImage(video,0,0);
              captured.push(canvas.toDataURL('image/jpeg',0.8).split(',')[1]);
              await new Promise(resolve => setTimeout(resolve,1000));
            }
            return captured;
          } finally {
            stream.getTracks().forEach(track => track.stop());
            video.remove();
          }
        })(%d)""" % max(1,min(WEBCAM_SECONDS,30)))
        webcam_rows = []
        fig, axes = plt.subplots(1,min(3,len(frames)),figsize=(12,4),squeeze=False)
        saved = False
        for index, encoded in enumerate(frames):
            frame = cv2.imdecode(np.frombuffer(base64.b64decode(encoded),dtype=np.uint8),cv2.IMREAD_COLOR)
            annotated, rows, _ = predict_frame(frame,index)
            webcam_rows.extend(rows)
            if index < axes.size:
                show_frame(axes[0,index],annotated,f"Camera frame {index}")
            if rows and not saved:
                saved = cv2.imwrite(str(output/"webcam_detection.jpg"),annotated)
        save_figure(fig,"webcam_preview.png")
        pd.DataFrame(webcam_rows,columns=columns).to_csv(output/"webcam_detections.csv",index=False)
    else:
        camera = cv2.VideoCapture(0)
        started = time.monotonic()
        snapshot_saved = False
        index = 0
        try:
            if not camera.isOpened():
                raise OSError("Could not open the local camera.")
            while WEBCAM_MODE == "window" or time.monotonic()-started < WEBCAM_SECONDS:
                ok, frame = camera.read()
                if not ok:
                    break
                annotated, rows, _ = predict_frame(frame,index)
                if WEBCAM_MODE == "window":
                    cv2.imshow("Threat Detection (q quit, s snapshot)",annotated)
                    key = cv2.waitKey(1)&0xff
                    if key == ord("q"):
                        break
                    if key == ord("s"):
                        cv2.imwrite(str(output/f"webcam_{index}.jpg"),annotated)
                else:
                    from IPython.display import Image as NotebookImage
                    clear_output(wait=True)
                    ok, encoded = cv2.imencode(".jpg",annotated)
                    if ok:
                        display(NotebookImage(data=encoded.tobytes()))
                    if rows and not snapshot_saved:
                        snapshot_saved = cv2.imwrite(str(output/"webcam_detection.jpg"),annotated)
                index += 1
        finally:
            camera.release()
            if WEBCAM_MODE == "window":
                cv2.destroyAllWindows()
else:
    print("Webcam off. Enable a mode in Section 1 to test a camera.")

## 7. Record and download
Enter observations for the eight scenarios in OBSERVATIONS in Section 1. Rerun the settings, test log and download cells.

In [ ]:
#@title Save the manual test log
scenarios = [
    ("Near distance", "Visible object at short range"),
    ("Far distance", "Object occupies a small part of the frame"),
    ("Dim light", "Object visible under low light"),
    ("Partial occlusion", "Part of the object is hidden"),
    ("Multiple objects", "Several visible objects"),
    ("Look-alike", "Ordinary object with a similar shape"),
    ("Empty scene", "No target objects"),
    ("Motion blur", "Target or camera is moving"),
]
log = pd.DataFrame([[name,condition,OBSERVATIONS.get(name,"")] for name,condition in scenarios],
                   columns=["scenario","condition","observed"])
display(log)
log.to_csv(output/"manual_test_log.csv",index=False)

In [ ]:
#@title 8. Download demo outputs
bundle = output / "threat_detection_demo.zip"
with ZipFile(bundle,"w") as archive:
    for result_file in sorted(output.rglob("*")):
        if result_file.is_file() and result_file != bundle:
            archive.write(result_file,arcname=result_file.relative_to(output))
if files is not None:
    files.download(str(bundle))
else:
    print(bundle.resolve())